# VeinForge — 训练强通用叶脉底座(免费 GPU)

Colab:`Runtime → Change runtime type → GPU (T4)`,然后从上到下运行。
用免费公共数据(LeafVeinCNN)+ GPU 训一个强通用脉分割底座;之后有小麦/大麦图就 `--init` 微调。

In [ ]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else '没有 GPU — 去选 GPU runtime')

In [ ]:
!git clone -q https://github.com/Vambrocop/VeinForge
%cd VeinForge
!pip install -q -e ".[dl]"

In [ ]:
# 下载+抽tile(全部约10GB;先4个plot验证,跑通后去掉 --plots 用全部)
!python scripts/fetch_leafveincnn.py --plots SLF BNTa DAF2 ESAa

In [ ]:
# 按叶片切 train/val,原分辨率 256
!python scripts/split_by_leaf.py --src data/leafvein --size 256 --val-frac 0.2

In [ ]:
# GPU 训练:原分辨率/batch16/多轮/clDice+增强/固定holdout存最佳
!python scripts/train_dl.py --data data/train --val data/val --epochs 100 --batch 16 --out models/leafvein_base.pt

In [ ]:
from google.colab import files
files.download('models/leafvein_base.pt')  # 下回本地做底座 --init 微调